# Stage 4B — Xception + Emotion Fusion Experiments (XGBoost)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier


In [2]:
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()

SUBSET_NAME = "final"

MANIFEST_PATH = PROJECT_ROOT / "exports/final/csv" / f"{SUBSET_NAME}_manifest_export.csv"
EMOTION_PATH = PROJECT_ROOT / "datasets/emotion_annotated/metadata" / f"{SUBSET_NAME}_video_emotion_features.csv"
XCEPTION_SCORES_PATH = PROJECT_ROOT / "outputs" / "deepfakebench_scores" / "ThesisFinal_xception_video_scores.csv"

MERGED_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_merged_xception_emotion.csv"
RANDOM_SEED = 42
TEST_SIZE = 0.2

print(MANIFEST_PATH)
print(EMOTION_PATH)
print(XCEPTION_SCORES_PATH)

/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/exports/final/csv/final_manifest_export.csv
/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/emotion_annotated/metadata/final_video_emotion_features.csv
/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/outputs/deepfakebench_scores/ThesisFinal_xception_video_scores.csv


In [3]:
manifest_df = pd.read_csv(MANIFEST_PATH)
emotion_df = pd.read_csv(EMOTION_PATH)
scores_df = pd.read_csv(XCEPTION_SCORES_PATH)

print(manifest_df.shape, emotion_df.shape, scores_df.shape)
display(scores_df.head())

(800, 20) (790, 56) (158, 4)


,video_id,detector_score,n_frames,label
0,Celeb-real__id10_0000__9273e8bc,0.131907,32,0
1,Celeb-real__id10_0002__6a16be4b,0.197790,32,0
2,Celeb-real__id10_0004__2f2a56d2,0.216124,32,0
3,Celeb-real__id10_0006__556711a5,0.286550,29,0
4,Celeb-real__id10_0007__841eee79,0.310433,32,0


In [4]:
if "label" in scores_df.columns:
    scores_df["label_from_detector"] = scores_df["label"].map({0: "real", 1: "fake"}).fillna(scores_df["label"])

scores_keep = ["video_id", "detector_score"]
if "label_from_detector" in scores_df.columns:
    scores_keep.append("label_from_detector")
if "n_frames" in scores_df.columns:
    scores_keep.append("n_frames")

scores_df = scores_df[scores_keep].copy()

df = manifest_df.merge(emotion_df, on="video_id", how="inner", suffixes=("", "_emo"))
df = df.merge(scores_df, on="video_id", how="inner")

df["y"] = df["label"].map({"real": 0, "fake": 1})

print(df.shape)
display(df.head())
df.to_csv(MERGED_OUT, index=False)
print(MERGED_OUT)

(155, 79)


,video_id,path,relative_path,split,label,source_subset,manipulation_family,manipulation_type,identity,source_identity,...,mean_score_sexual_lust,mean_score_shame,mean_score_sourness,mean_score_teasing,mean_score_thankfulness_gratitude,mean_score_triumph,detector_score,label_from_detector,n_frames,y
0,Celeb-real__id50_0005__3870d23c,videos/real/Celeb-real/id50_0005.mp4,Celeb-real/id50_0005.mp4,test,real,Celeb-real,NaN,Celeb-real,id50,id50,...,1.225383,-0.432930,-0.560299,0.632307,1.407119,-0.901347,0.492663,real,32,0
1,Celeb-synthesis__FaceReenact__TPSMM__id23_id19...,videos/fake/FaceReenact/id23_id19_0000.mp4,Celeb-synthesis/FaceReenact/TPSMM/id23_id19_00...,test,fake,Celeb-synthesis,FaceReenact,TPSMM,id23,id23,...,1.685264,-0.440217,-0.736206,0.933438,1.848022,-0.586169,0.871961,fake,32,1
2,Celeb-synthesis__FaceReenact__LivePortrait__id...,videos/fake/FaceReenact/id50_id54_0006.mp4,Celeb-synthesis/FaceReenact/LivePortrait/id50_...,test,fake,Celeb-synthesis,FaceReenact,LivePortrait,id50,id50,...,2.225306,-0.328005,-0.612466,0.549861,0.839380,-0.541393,0.268077,fake,32,1
3,Celeb-synthesis__FaceSwap__SimSwap__id23_id31_...,videos/fake/FaceSwap/id23_id31_0002.mp4,Celeb-synthesis/FaceSwap/SimSwap/id23_id31_000...,test,fake,Celeb-synthesis,FaceSwap,SimSwap,id23,id23,...,1.805201,-0.487330,-0.651732,0.730131,1.210467,-0.485849,0.169931,fake,32,1
4,Celeb-synthesis__FaceReenact__TPSMM__id50_id57...,videos/fake/FaceReenact/id50_id57_0001.mp4,Celeb-synthesis/FaceReenact/TPSMM/id50_id57_00...,test,fake,Celeb-synthesis,FaceReenact,TPSMM,id50,id50,...,1.884591,-0.532768,-0.610328,1.136294,2.191024,0.267602,0.699516,fake,32,1


/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_merged_xception_emotion.csv


In [5]:
def binary_metrics(y_true, y_score, thr=0.5):
    y_pred = (y_score >= thr).astype(int)
    return {
        "AUC": roc_auc_score(y_true, y_score),
        "ACC": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
    }

def evaluate_group_auc(df_in, group_col):
    rows = []
    for group_value, sub in df_in.groupby(group_col):
        if sub["y"].nunique() < 2:
            continue
        rows.append({
            group_col: group_value,
            "n": len(sub),
            "AUC": roc_auc_score(sub["y"], sub["detector_score"])
        })
    return pd.DataFrame(rows).sort_values("AUC", ascending=False)

In [6]:
baseline_metrics = binary_metrics(df["y"].values, df["detector_score"].values)
baseline_metrics

{'AUC': 0.6742171885409726,
 'ACC': 0.6709677419354839,
 'F1': 0.5321100917431193,
 'Precision': 0.8787878787878788,
 'Recall': 0.3815789473684211}

In [7]:
emotion_auc_df = evaluate_group_auc(df, "dominant_emotion")
display(emotion_auc_df)

df["arousal_bin"] = pd.qcut(df["mean_arousal"], q=3, labels=["low", "mid", "high"], duplicates="drop")
arousal_auc_df = evaluate_group_auc(df, "arousal_bin")
display(arousal_auc_df)

,dominant_emotion,n,AUC
4,thankfulness_gratitude,2,1.000000
1,astonishment_surprise,51,0.711420
2,contentment,53,0.670940
3,elation,30,0.634259
0,anger,8,0.625000


,arousal_bin,n,AUC
1,mid,51,0.750000
2,high,52,0.740741
0,low,52,0.598438


In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df["y"]
)

print(train_df.shape, test_df.shape)

(124, 80) (31, 80)


In [9]:
emotion_features = [
    "mean_valence",
    "std_valence",
    "mean_arousal",
    "std_arousal",
    "max_arousal",
    "emotion_entropy",
    "transition_rate",
    "arousal_variation",
    "neutral_ratio",
]

fusion_features = ["detector_score"] + emotion_features

if "dominant_emotion" in df.columns:
    dummies_train = pd.get_dummies(train_df["dominant_emotion"], prefix="emo")
    dummies_test = pd.get_dummies(test_df["dominant_emotion"], prefix="emo")
    dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

    train_df_model = pd.concat([train_df.reset_index(drop=True), dummies_train.reset_index(drop=True)], axis=1)
    test_df_model = pd.concat([test_df.reset_index(drop=True), dummies_test.reset_index(drop=True)], axis=1)

    dummy_cols = list(dummies_train.columns)
    emotion_features = emotion_features + dummy_cols
    fusion_features = ["detector_score"] + emotion_features
else:
    train_df_model = train_df.copy()
    test_df_model = test_df.copy()

print(emotion_features)
print(fusion_features)

['mean_valence', 'std_valence', 'mean_arousal', 'std_arousal', 'max_arousal', 'emotion_entropy', 'transition_rate', 'arousal_variation', 'neutral_ratio', 'emo_amusement', 'emo_anger', 'emo_astonishment_surprise', 'emo_contentment', 'emo_disappointment', 'emo_elation', 'emo_interest', 'emo_sexual_lust', 'emo_thankfulness_gratitude']
['detector_score', 'mean_valence', 'std_valence', 'mean_arousal', 'std_arousal', 'max_arousal', 'emotion_entropy', 'transition_rate', 'arousal_variation', 'neutral_ratio', 'emo_amusement', 'emo_anger', 'emo_astonishment_surprise', 'emo_contentment', 'emo_disappointment', 'emo_elation', 'emo_interest', 'emo_sexual_lust', 'emo_thankfulness_gratitude']


In [10]:
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ))
])

X_train_base = train_df_model[["detector_score"]]
X_test_base = test_df_model[["detector_score"]]
y_train = train_df_model["y"]
y_test = test_df_model["y"]

baseline_model.fit(X_train_base, y_train)
baseline_model_scores = baseline_model.predict_proba(X_test_base)[:, 1]

baseline_model_metrics = binary_metrics(y_test.values, baseline_model_scores)
baseline_model_metrics


{'AUC': 0.5770833333333333,
 'ACC': 0.5161290322580645,
 'F1': 0.4827586206896552,
 'Precision': 0.5,
 'Recall': 0.4666666666666667}

In [11]:
emotion_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ))
])

X_train_emotion = train_df_model[emotion_features]
X_test_emotion = test_df_model[emotion_features]

emotion_model.fit(X_train_emotion, y_train)
emotion_model_scores = emotion_model.predict_proba(X_test_emotion)[:, 1]

emotion_model_metrics = binary_metrics(y_test.values, emotion_model_scores)
emotion_model_metrics


{'AUC': 0.5208333333333334,
 'ACC': 0.5483870967741935,
 'F1': 0.46153846153846156,
 'Precision': 0.5454545454545454,
 'Recall': 0.4}

In [12]:
fusion_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ))
])

X_train_fusion = train_df_model[fusion_features]
X_test_fusion = test_df_model[fusion_features]

fusion_model.fit(X_train_fusion, y_train)
fusion_scores = fusion_model.predict_proba(X_test_fusion)[:, 1]

fusion_metrics = binary_metrics(y_test.values, fusion_scores)
fusion_metrics


{'AUC': 0.675,
 'ACC': 0.6129032258064516,
 'F1': 0.5714285714285714,
 'Precision': 0.6153846153846154,
 'Recall': 0.5333333333333333}

In [13]:
results_df = pd.DataFrame([
    {"model": "xception_score_only_raw", **binary_metrics(df["y"].values, df["detector_score"].values)},
    {"model": "baseline_only_xgboost", **baseline_model_metrics},
    {"model": "emotion_only_xgboost", **emotion_model_metrics},
    {"model": "fusion_xgboost", **fusion_metrics},
])

display(results_df.sort_values("AUC", ascending=False))


,model,AUC,ACC,F1,Precision,Recall
3,fusion_xgboost,0.675000,0.612903,0.571429,0.615385,0.533333
0,xception_score_only_raw,0.674217,0.670968,0.532110,0.878788,0.381579
1,baseline_only_xgboost,0.577083,0.516129,0.482759,0.500000,0.466667
2,emotion_only_xgboost,0.520833,0.548387,0.461538,0.545455,0.400000


In [14]:
ablation_sets = {
    "detector_plus_affect": ["detector_score", "mean_valence", "mean_arousal", "max_arousal"],
    "detector_plus_dynamics": ["detector_score", "emotion_entropy", "transition_rate", "arousal_variation"],
    "detector_plus_all": fusion_features,
}

ablation_rows = []

for name, feats in ablation_sets.items():
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ))
    ])
    model.fit(train_df_model[feats], y_train)
    scores = model.predict_proba(test_df_model[feats])[:, 1]
    ablation_rows.append({"ablation": name, **binary_metrics(y_test.values, scores)})

ablation_df = pd.DataFrame(ablation_rows).sort_values("AUC", ascending=False)
display(ablation_df)


,ablation,AUC,ACC,F1,Precision,Recall
2,detector_plus_all,0.675000,0.612903,0.571429,0.615385,0.533333
0,detector_plus_affect,0.633333,0.612903,0.625000,0.588235,0.666667
1,detector_plus_dynamics,0.554167,0.548387,0.533333,0.533333,0.533333


In [15]:
RESULTS_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_fusion_xgboost_results.csv"
ABLATION_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_xgboost_ablation_results.csv"
EMOTION_AUC_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_auc_by_emotion.csv"
AROUSAL_AUC_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_auc_by_arousal.csv"

results_df.to_csv(RESULTS_OUT, index=False)
ablation_df.to_csv(ABLATION_OUT, index=False)
emotion_auc_df.to_csv(EMOTION_AUC_OUT, index=False)
arousal_auc_df.to_csv(AROUSAL_AUC_OUT, index=False)

print(RESULTS_OUT)
print(ABLATION_OUT)
print(EMOTION_AUC_OUT)
print(AROUSAL_AUC_OUT)


/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_xception_fusion_xgboost_results.csv
/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_xception_xgboost_ablation_results.csv
/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_xception_auc_by_emotion.csv
/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_xception_auc_by_arousal.csv
